# LLM Fine-Tuning v2 with Starlar SFT Dataset

This notebook fine-tunes Mistral-7B-Instruct-v0.2 using the Starlar LLM SFT v2 dataset.

Main differences from the previous LLM fine-tuning:
- Uses Starlar `llm.jsonl` message-based dataset.
- Uses cleaner source-grounded assistant answers.
- Uses assistant-only loss masking.
- Keeps checkpoints every 50 steps.
- Saves a new LoRA adapter separately from the previous model.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime > Change runtime type > T4 GPU seç.")

CUDA available: True
GPU: NVIDIA L4


In [18]:
import os

project_path = "/content/drive/MyDrive/turkish_legal_rag"

processed_path = f"{project_path}/data/processed"
outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"

train_jsonl_path = f"{processed_path}/starlar_llm_sft_v2_train.jsonl"
val_jsonl_path = f"{processed_path}/starlar_llm_sft_v2_val.jsonl"
test_jsonl_path = f"{processed_path}/starlar_llm_sft_v2_test.jsonl"

adapter_output_path = f"{models_path}/mistral_legal_qlora_starlar_v2_800steps"

print("Train exists:", os.path.exists(train_jsonl_path))
print("Val exists:", os.path.exists(val_jsonl_path))
print("Test exists:", os.path.exists(test_jsonl_path))
print("Adapter output:", adapter_output_path)

Train exists: True
Val exists: True
Test exists: True
Adapter output: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_800steps


In [19]:
for path in [train_jsonl_path, val_jsonl_path, test_jsonl_path]:
    print(path)
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Size MB:", round(os.path.getsize(path) / (1024 * 1024), 2))
    print("-" * 80)

/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_train.jsonl
Exists: True
Size MB: 9.43
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_val.jsonl
Exists: True
Size MB: 1.23
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_test.jsonl
Exists: True
Size MB: 1.19
--------------------------------------------------------------------------------


In [20]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets

In [21]:
import os
import json
import gc
import math
import pandas as pd
import numpy as np
import torch

from dataclasses import dataclass
from typing import Dict, List, Any

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

In [22]:
data_files = {
    "train": train_jsonl_path,
    "validation": val_jsonl_path,
    "test": test_jsonl_path
}

dataset = load_dataset("json", data_files=data_files)

print(dataset)

print("\nTrain sample keys:")
print(dataset["train"][0].keys())

print("\nSample text:")
print(dataset["train"][0]["text"][:2000])

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'system_content', 'user_content', 'assistant_content', 'metadata_source', 'metadata_variant_type', 'metadata_source_id'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['id', 'text', 'system_content', 'user_content', 'assistant_content', 'metadata_source', 'metadata_variant_type', 'metadata_source_id'],
        num_rows: 400
    })
    test: Dataset({
        features: ['id', 'text', 'system_content', 'user_content', 'assistant_content', 'metadata_source', 'metadata_variant_type', 'metadata_source_id'],
        num_rows: 400
    })
})

Train sample keys:
dict_keys(['id', 'text', 'system_content', 'user_content', 'assistant_content', 'metadata_source', 'metadata_variant_type', 'metadata_source_id'])

Sample text:
<s>[INST] Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citatio

In [23]:
from huggingface_hub import login

login()

In [24]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# Kritik: prompt çok uzunsa baştan kesilsin, [/INST] + cevap kısmı kalsın
tokenizer.truncation_side = "left"

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("Padding side:", tokenizer.padding_side)
print("Truncation side:", tokenizer.truncation_side)

Tokenizer loaded.
Pad token: </s>
Padding side: right
Truncation side: left


In [25]:
MAX_LENGTH = 1024
RESPONSE_MARKER = "[/INST]"


def tokenize_with_assistant_only_labels(example):
    text = str(example["text"])

    if RESPONSE_MARKER not in text:
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            add_special_tokens=False
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        labels = input_ids.copy()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

    prompt_part = text.split(RESPONSE_MARKER)[0] + RESPONSE_MARKER
    assistant_part = text[len(prompt_part):]

    prompt_ids = tokenizer(
        prompt_part,
        add_special_tokens=False
    )["input_ids"]

    assistant_ids = tokenizer(
        assistant_part,
        add_special_tokens=False
    )["input_ids"]

    # Eğer toplam uzunluk max_length'i geçerse prompt başından kes.
    total_len = len(prompt_ids) + len(assistant_ids)

    if total_len > MAX_LENGTH:
        overflow = total_len - MAX_LENGTH

        if overflow < len(prompt_ids):
            prompt_ids = prompt_ids[overflow:]
        else:
            # Aşırı uzun durumda assistant cevabını korumaya çalış.
            prompt_ids = []
            assistant_ids = assistant_ids[-MAX_LENGTH:]

    input_ids = prompt_ids + assistant_ids
    attention_mask = [1] * len(input_ids)

    # Prompt kısmını loss dışı bırakıyoruz.
    labels = [-100] * len(prompt_ids) + assistant_ids.copy()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [26]:
tokenized_dataset = dataset.map(
    tokenize_with_assistant_only_labels,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing with assistant-only labels"
)

print(tokenized_dataset)

sample = tokenized_dataset["train"][0]

print("input_ids len:", len(sample["input_ids"]))
print("labels len:", len(sample["labels"]))
print("loss token count:", sum([1 for x in sample["labels"] if x != -100]))

decoded_input = tokenizer.decode(sample["input_ids"], skip_special_tokens=False)
print("\nDecoded input preview:")
print(decoded_input[:2000])

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 400
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 400
    })
})
input_ids len: 678
labels len: 678
loss token count: 185

Decoded input preview:
<s>[INST] Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citation bilgisini mutlaka belirt. Kaynak kesin resmî karar metni değilse bunu kesin hüküm gibi sunma.

[Kaynak]
Başlık: Türk Medenî Kanunu
Kaynak: TURKISH_LAW_ESKI_LOW_RISK_ONLY
Dosya: TURKISH_LAW_ESKI_clean_source_for_chunking_LOW_RISK_ONLY.txt
Chunk ID: turkish_law_eski_4721_turk_medeni_kanunu_m958
Citation: TURKISH_LAW_ESKI_LOW_RISK_ONLY - Türk Medenî Kanunu m.958 - turkish_law_eski

In [27]:
@dataclass
class CausalLMDataCollatorWithLabelPadding:
    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_length = max(len(f["input_ids"]) for f in features)

        input_ids_batch = []
        attention_mask_batch = []
        labels_batch = []

        pad_token_id = self.tokenizer.pad_token_id

        for f in features:
            input_ids = f["input_ids"]
            attention_mask = f["attention_mask"]
            labels = f["labels"]

            pad_length = max_length - len(input_ids)

            input_ids_batch.append(input_ids + [pad_token_id] * pad_length)
            attention_mask_batch.append(attention_mask + [0] * pad_length)
            labels_batch.append(labels + [self.label_pad_token_id] * pad_length)

        return {
            "input_ids": torch.tensor(input_ids_batch, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_batch, dtype=torch.long),
            "labels": torch.tensor(labels_batch, dtype=torch.long)
        }


data_collator = CausalLMDataCollatorWithLabelPadding(tokenizer=tokenizer)

print("Custom data collator ready.")

Custom data collator ready.


In [28]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

print("Base model loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base model loaded.


In [29]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


In [30]:
os.makedirs(adapter_output_path, exist_ok=True)

MAX_STEPS = 800
SAVE_STEPS = 50
EVAL_STEPS = 50

training_args = TrainingArguments(
    output_dir=adapter_output_path,

    max_steps=MAX_STEPS,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # 800 step için 1e-4 daha güvenli
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # Önceki bf16/fp16 scaler hatalarını yememek için kapalı
    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=6,

    # En iyi eval_loss checkpointini final model olarak seç
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",

    remove_unused_columns=False
)

print("Training args ready.")
print("Max steps:", MAX_STEPS)
print("Save every:", SAVE_STEPS)
print("Eval every:", EVAL_STEPS)
print("Learning rate:", training_args.learning_rate)
print("Output:", adapter_output_path)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training args ready.
Max steps: 800
Save every: 50
Eval every: 50
Learning rate: 0.0001
Output: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_800steps


In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=4,
            early_stopping_threshold=0.0
        )
    ]
)

print("Trainer ready.")

Trainer ready.


In [32]:
train_result = trainer.train()

print("Training finished.")

Step,Training Loss,Validation Loss
50,0.006121,0.004296
100,0.000664,0.002159
150,0.001800,0.001965
200,0.005546,0.002857
250,0.007028,0.001719
300,0.004848,0.001810
350,0.006124,0.001616
400,0.009076,0.001352
450,0.002134,0.001191
500,0.000384,0.001210


Training finished.


In [33]:
trainer.save_model(adapter_output_path)
tokenizer.save_pretrained(adapter_output_path)

print("Adapter saved to:", adapter_output_path)

print("\nFiles in adapter folder:")
for item in os.listdir(adapter_output_path):
    print("-", item)

Adapter saved to: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_800steps

Files in adapter folder:
- checkpoint-550
- checkpoint-600
- checkpoint-650
- checkpoint-700
- checkpoint-750
- checkpoint-800
- README.md
- adapter_model.safetensors
- adapter_config.json
- chat_template.jinja
- tokenizer_config.json
- tokenizer.json
- training_args.bin


In [34]:
required_files = [
    "adapter_config.json",
    "adapter_model.safetensors"
]

for file in required_files:
    path = os.path.join(adapter_output_path, file)
    print(file, "exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Size MB:", round(os.path.getsize(path) / (1024 * 1024), 2))

adapter_config.json exists: True
Size MB: 0.0
adapter_model.safetensors exists: True
Size MB: 160.06


In [35]:
log_df = pd.DataFrame(trainer.state.log_history)

display(log_df.tail(30))

log_path = f"{metrics_path}/mistral_legal_qlora_starlar_v2_800steps_training_log.csv"

log_df.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

print("Training log saved:", log_path)

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
67,0.003199,0.001769,2.031675e-05,1.425,570,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
68,0.003756,0.001918,1.871260e-05,1.450,580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,0.000717,0.001172,1.715972e-05,1.475,590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,0.002024,0.003228,1.566066e-05,1.500,600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71,NaN,NaN,NaN,1.500,600,0.001196,230.1674,1.738,1.738,NaN,NaN,NaN,NaN,NaN
72,0.002243,0.006784,1.421788e-05,1.525,610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73,0.008998,0.000969,1.283373e-05,1.550,620,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,0.008633,0.000910,1.151049e-05,1.575,630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,0.003904,0.002421,1.025033e-05,1.600,640,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,0.006136,0.089186,9.055302e-06,1.625,650,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Training log saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_800steps_training_log.csv


In [36]:
summary_df = pd.DataFrame([{
    "base_model": model_name,
    "method": "QLoRA",
    "dataset": "Starlar LLM SFT v2",
    "train_file": train_jsonl_path,
    "val_file": val_jsonl_path,
    "train_rows": len(dataset["train"]),
    "val_rows": len(dataset["validation"]),
    "max_steps": MAX_STEPS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "max_length": MAX_LENGTH,
    "assistant_only_loss": True,
    "learning_rate": training_args.learning_rate,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "per_device_train_batch_size": training_args.per_device_train_batch_size,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "adapter_output_path": adapter_output_path,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric
}])

summary_path = f"{metrics_path}/mistral_legal_qlora_starlar_v2_800steps_summary.csv"

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

display(summary_df)

print("Summary saved:", summary_path)

,base_model,method,dataset,train_file,val_file,train_rows,val_rows,max_steps,save_steps,eval_steps,...,assistant_only_loss,learning_rate,gradient_accumulation_steps,per_device_train_batch_size,lora_r,lora_alpha,lora_dropout,adapter_output_path,best_model_checkpoint,best_metric
0,mistralai/Mistral-7B-Instruct-v0.2,QLoRA,Starlar LLM SFT v2,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,3200,400,800,50,50,...,True,0.0001,8,1,16,32,0.05,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...,0.001149


Summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_800steps_summary.csv


In [37]:
def extract_prompt_from_sft_text(text):
    text = str(text)

    if RESPONSE_MARKER in text:
        return text.split(RESPONSE_MARKER)[0] + RESPONSE_MARKER

    return text


def generate_answer_only(model, tokenizer, prompt, max_new_tokens=160, max_length=1024):
    tokenizer.truncation_side = "left"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()

In [38]:
sample_text = dataset["test"][0]["text"]
sample_prompt = extract_prompt_from_sft_text(sample_text)

expected_answer = dataset["test"][0]["assistant_content"]

generated_answer = generate_answer_only(
    model=model,
    tokenizer=tokenizer,
    prompt=sample_prompt,
    max_new_tokens=160,
    max_length=1024
)

print("PROMPT LAST PART:")
print(sample_prompt[-1500:])

print("\nEXPECTED:")
print(expected_answer)

print("\nGENERATED:")
print(generated_answer)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


PROMPT LAST PART:
<s>[INST] Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citation bilgisini mutlaka belirt. Kaynak kesin resmî karar metni değilse bunu kesin hüküm gibi sunma.

[Kaynak]
Başlık: Genel hukuk / sınıflandırma bekliyor
Kaynak: ORICON
Dosya: ORICON_clean_legal_source_for_chunking.txt
Chunk ID: oricon_genel_001801
Citation: ORICON - Genel hukuk / sınıflandırma bekliyor - oricon_genel_001801
Metin: Bir yıllık sürenin başlangıç tarihi, eşlerin resmi nikâh yaptıkları tarih esas alınarak belirlenir.

Soru: Bu metinden hareketle 'yıllık sürenin başlangıç tarihi' konusunu açıklarken nelere dayanmalıyım? [/INST]

EXPECTED:
Bu açıklama şu kaynak bilgilerine dayandırılmalıdır: Kategori: Genel hukuk / sınıflandırma bekliyor. Kaynak metindeki esas içerik şudur: Bir yıllık sürenin başlangıç tarihi, eşlerin resmi nikâh yaptıkları tarih esas alınarak belirlenir.

Kaynak: 

In [39]:
files_to_check = [
    log_path,
    summary_path,
    os.path.join(adapter_output_path, "adapter_config.json"),
    os.path.join(adapter_output_path, "adapter_model.safetensors")
]

print("FINAL CHECK")
print("=" * 80)

for file in files_to_check:
    print(file)
    print("Exists:", os.path.exists(file))
    if os.path.exists(file):
        print("Size KB:", round(os.path.getsize(file) / 1024, 2))
    print("-" * 80)

print("Done.")

FINAL CHECK
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_800steps_training_log.csv
Exists: True
Size KB: 7.67
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_starlar_v2_800steps_summary.csv
Exists: True
Size KB: 0.78
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_800steps/adapter_config.json
Exists: True
Size KB: 1.09
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_starlar_v2_800steps/adapter_model.safetensors
Exists: True
Size KB: 163898.67
--------------------------------------------------------------------------------
Done.
